In [2]:
# Install libraries
!pip install catboost

# Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv("application_train.csv")

# Check data
print(df.head())
print(df.columns)

# Fill missing numeric values
df.fillna(df.mean(numeric_only=True), inplace=True)

# Fill missing categorical values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# Convert categorical columns into numeric
df = pd.get_dummies(df, drop_first=True)

# Target column
# Home Credit dataset target column = TARGET
X = df.drop("TARGET", axis=1)
y = df["TARGET"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale data for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -----------------------------------
# Logistic Regression Model
# -----------------------------------
lr_model = LogisticRegression(max_iter=5000) # Increased max_iter
lr_model.fit(X_train_scaled, y_train)

lr_prob = lr_model.predict_proba(X_test_scaled)[:,1]

# Threshold optimization
threshold = 0.4
lr_pred = (lr_prob >= threshold).astype(int)

# Evaluation
cm_lr = confusion_matrix(y_test, lr_pred)

print("Logistic Regression Confusion Matrix:")
print(cm_lr)

print("Logistic Regression Accuracy:")
print(accuracy_score(y_test, lr_pred))

# Business Cost Calculation
# Example:
# False Positive = 100
# False Negative = 500

fp_lr = cm_lr[0][1] * 100
fn_lr = cm_lr[1][0] * 500

total_cost_lr = fp_lr + fn_lr

print("Logistic Regression Total Cost:", total_cost_lr)

# -----------------------------------
# CatBoost Model
# -----------------------------------
cat_model = CatBoostClassifier(verbose=0)
cat_model.fit(X_train, y_train)

cat_prob = cat_model.predict_proba(X_test)[:,1]

# Threshold optimization
cat_pred = (cat_prob >= threshold).astype(int)

# Evaluation
cm_cat = confusion_matrix(y_test, cat_pred)

print("\nCatBoost Confusion Matrix:")
print(cm_cat)

print("CatBoost Accuracy:")
print(accuracy_score(y_test, cat_pred))

# Business Cost Calculation
fp_cat = cm_cat[0][1] * 100
fn_cat = cm_cat[1][0] * 500

total_cost_cat = fp_cat + fn_cat

print("CatBoost Total Cost:", total_cost_cat)

# -----------------------------------
# Final Comparison
# -----------------------------------
results = pd.DataFrame({
    "Model": ["Logistic Regression", "CatBoost"],
    "Accuracy": [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, cat_pred)
    ],
    "Total Cost": [
        total_cost_lr,
        total_cost_cat
    ]
})

print("\nFinal Comparison:")
print(results)

   SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR  \
0      100002       1         Cash loans           M            N   
1      100003       0         Cash loans           F            N   
2      100004       0    Revolving loans           M            Y   
3      100006       0         Cash loans           F            N   
4      100007       0         Cash loans           M            N   

  FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  \
0               Y             0          202500.0    406597.5      24700.5   
1               N             0          270000.0   1293502.5      35698.5   
2               Y             0           67500.0    135000.0       6750.0   
3               Y             0          135000.0    312682.5      29686.5   
4               Y             0          121500.0    513000.0      21865.5   

   ...  FLAG_DOCUMENT_18 FLAG_DOCUMENT_19 FLAG_DOCUMENT_20 FLAG_DOCUMENT_21  \
0  ...                 0             